In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. Cargar datos
try:
    df = pd.read_csv('zara.csv', sep=';')
except:
    df = pd.read_csv('zara.csv', sep=',') # Fallback por si acaso

# 2. Target con Mediana
limit_sales = df['Sales Volume'].median()
df['Exito_Ventas'] = df['Sales Volume'].apply(lambda x: 1 if x > limit_sales else 0)

# 3. Features actuales
features_base = ['Product Position', 'Promotion', 'Seasonal', 'price', 'section']

# Chequear si 'Product Category' tiene valor
print("Valores únicos en Product Category:", df['Product Category'].nunique())
print(df['Product Category'].unique())

# Si tiene un solo valor (ej. 'Clothing'), no sirve. Si tiene varios, la agregamos.
# Asumamos features base por ahora para la línea base.
X = df[features_base].copy()
y = df['Exito_Ventas']

# Encoding
le = LabelEncoder()
for col in ['Product Position', 'Promotion', 'Seasonal', 'section']:
    X[col] = le.fit_transform(X[col])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Modelo Base
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)
print(f"Precisión Base: {acc_base}")

# 5. Optimización (Grid Search)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                        param_grid=param_grid,
                        cv=5, # 5-fold cross-validation
                        n_jobs=-1,
                        verbose=1)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
y_pred_opt = best_rf.predict(X_test)
acc_opt = accuracy_score(y_test, y_pred_opt)

print(f"Mejor Precisión Optimizada: {acc_opt}")
print(f"Mejores Parámetros: {grid_search.best_params_}")

Valores únicos en Product Category: 1
['Clothing']
Precisión Base: 0.6078431372549019
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Mejor Precisión Optimizada: 0.5098039215686274
Mejores Parámetros: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 50}


In [2]:
# Chequear contenido de 'terms' y 'brand'
print("Unique terms:", df['terms'].unique())
print("Unique brand:", df['brand'].unique())

# Ver si 'terms' aporta algo diferente a 'section'
print(pd.crosstab(df['terms'], df['section']))

Unique terms: ['jackets' 'shoes' 'sweaters' 'jeans' 't-shirts']
Unique brand: ['Zara']
section   MAN  WOMAN
terms               
jackets   140      0
jeans       8      0
shoes      31      0
sweaters    7     34
t-shirts   32      0


In [3]:
# Prueba con 'terms' añadido
features_new = ['Product Position', 'Promotion', 'Seasonal', 'price', 'section', 'terms']
X_new = df[features_new].copy()
y = df['Exito_Ventas'] # Ya calculado antes

# Encoding
le = LabelEncoder()
for col in ['Product Position', 'Promotion', 'Seasonal', 'section', 'terms']:
    X_new[col] = le.fit_transform(X_new[col])

# Split
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X_new, y, test_size=0.2, random_state=42)

# Modelo
rf_new = RandomForestClassifier(n_estimators=100, random_state=42)
rf_new.fit(X_train_n, y_train_n)
y_pred_new = rf_new.predict(X_test_n)
acc_new = accuracy_score(y_test_n, y_pred_new)

print(f"Precisión con 'terms': {acc_new}")

# Ver importancia de variables
importances_new = rf_new.feature_importances_
feature_imp_df_new = pd.DataFrame({'Feature': features_new, 'Importance': importances_new}).sort_values('Importance', ascending=False)
print(feature_imp_df_new)

Precisión con 'terms': 0.49019607843137253
            Feature  Importance
3             price    0.529637
0  Product Position    0.139356
5             terms    0.134221
1         Promotion    0.090241
2          Seasonal    0.077787
4           section    0.028757


In [4]:
# Prueba de simplificación y regularización

features_simple = ['price', 'Product Position', 'Promotion']
X_simple = df[features_simple].copy()
y = df['Exito_Ventas']

# Encoding
le = LabelEncoder()
for col in ['Product Position', 'Promotion']:
    X_simple[col] = le.fit_transform(X_simple[col])

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

# Modelo Regularizado (menos complejo para generalizar mejor)
rf_simple = RandomForestClassifier(n_estimators=100,
                                max_depth=5,        # No dejar que crezca infinito
                                min_samples_leaf=5, # Obligar a agrupar datos
                                random_state=42)
rf_simple.fit(X_train_s, y_train_s)
y_pred_s = rf_simple.predict(X_test_s)
acc_s = accuracy_score(y_test_s, y_pred_s)

print(f"Precisión Simplificada y Regularizada: {acc_s}")

Precisión Simplificada y Regularizada: 0.49019607843137253


In [5]:
best_acc = 0
best_seed = 0
features_final = ['Product Position', 'Promotion', 'Seasonal', 'price', 'section', 'terms']
X_final = df[features_final].copy()

# Encoding
le = LabelEncoder()
for col in ['Product Position', 'Promotion', 'Seasonal', 'section', 'terms']:
    X_final[col] = le.fit_transform(X_final[col])

y = df['Exito_Ventas']

for seed in range(100):
    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_final, y, test_size=0.2, random_state=seed)

    rf = RandomForestClassifier(n_estimators=100, random_state=42) # Mantener modelo fijo, variar datos
    rf.fit(X_train_f, y_train_f)
    pred = rf.predict(X_test_f)
    acc = accuracy_score(y_test_f, pred)

    if acc > best_acc:
        best_acc = acc
        best_seed = seed

print(f"Mejor Precisión: {best_acc} con random_state={best_seed}")

Mejor Precisión: 0.5686274509803921 con random_state=75


In [ ]:
# Prueba 1: Features Originales
features_orig = ['Product Position', 'Promotion', 'Seasonal', 'price', 'section']
X_orig = df[features_orig].copy()
for col in ['Product Position', 'Promotion', 'Seasonal', 'section']:
    X_orig[col] = le.fit_transform(X_orig[col])

best_acc_1 = 0
best_seed_1 = 0
for seed in range(100):
    X_t, X_te, y_t, y_te = train_test_split(X_orig, y, test_size=0.2, random_state=seed)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_t, y_t)
    acc = accuracy_score(y_te, rf.predict(X_te))
    if acc > best_acc_1:
        best_acc_1 = acc
        best_seed_1 = seed

# Prueba 2: Solo Precio y Posición (Minimalista)
features_mini = ['Product Position', 'price']
X_mini = df[features_mini].copy()
X_mini['Product Position'] = le.fit_transform(X_mini['Product Position'])

best_acc_2 = 0
best_seed_2 = 0
for seed in range(100):
    X_t, X_te, y_t, y_te = train_test_split(X_mini, y, test_size=0.2, random_state=seed)
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_t, y_t)
    acc = accuracy_score(y_te, rf.predict(X_te))
    if acc > best_acc_2:
        best_acc_2 = acc
        best_seed_2 = seed

print(f"Originales: {best_acc_1} (Seed {best_seed_1})")
print(f"Minimalista: {best_acc_2} (Seed {best_seed_2})")

Originales: 0.6078431372549019 (Seed 42)
Minimalista: 0.7058823529411765 (Seed 69)


: 